# NB05 — Hyperparameter Tuning (Optuna)

**Project:** The Access Gap
**Phase:** CRISP-DM Phase 5 — Modelling (Optimisation)
**Inputs:** NB03 arrays, NB04 manifest
**Outputs:** Tuned models (`*_tuned.joblib`), best params JSON, comparison figures

---

## Objectives

Tune the **top 3 models** selected in NB04 with **Optuna TPE** sampler.
Objective = **5-fold CV F1** on raw (non-SMOTE) training + `class_weight='balanced'`.

| Step | Details |
|------|---------|
| Sampler | TPE (Tree-structured Parzen Estimator) |
| Budget | `N_TRIALS = 50` per model — increase to 100 for production |
| CV data | `X_train_pp_fe` + `y_train` — no SMOTE leakage across folds |
| Final fit | `X_train_smote_fe` after best params found |
| Evaluate | Validation set only — **test set sealed until NB06** |


In [ ]:
import subprocess, sys
REQUIRED = {
    'pyarrow':'pyarrow', 'pandas':'pandas', 'numpy':'numpy',
    'scikit-learn':'sklearn', 'imbalanced-learn':'imblearn',
    'xgboost':'xgboost', 'lightgbm':'lightgbm',
    'optuna':'optuna', 'matplotlib':'matplotlib',
    'seaborn':'seaborn', 'joblib':'joblib',
}
for pkg, imp in REQUIRED.items():
    try: __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('Packages OK')


---
## 0. Setup

In [ ]:
import sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import optuna
from optuna.samplers import TPESampler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import config as cfg
from evaluation import evaluate_model, find_optimal_threshold

plt.rcParams['figure.dpi']        = cfg.FIGURE_DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
PROCESSED = cfg.DATA_PROCESSED

print(f'Optuna  : {optuna.__version__}')
print(f'Root    : {PROJECT_ROOT}')


---
## 1. Load Data & NB04 Configuration

In [ ]:
X_train_smote = np.load(PROCESSED / 'X_train_smote_fe.npy')
y_train_smote = np.load(PROCESSED / 'y_train_smote_fe.npy')
X_train_pp    = np.load(PROCESSED / 'X_train_pp_fe.npy')    # non-SMOTE, for CV
X_val_pp      = np.load(PROCESSED / 'X_val_pp_fe.npy')
# X_test_pp — SEALED until NB06

y_train = pd.read_parquet(PROCESSED / 'y_train.parquet').squeeze().values
y_val   = pd.read_parquet(PROCESSED / 'y_val.parquet').squeeze().values

with open(PROCESSED / 'nb04_manifest.json') as f:         nb04_manifest = json.load(f)
with open(PROCESSED / 'val_results_nb04.json') as f:      nb04_val      = json.load(f)
with open(PROCESSED / 'threshold_results_nb04.json') as f: nb04_thr     = json.load(f)

TOP3       = nb04_manifest['top3_for_tuning']
SAFE_NAMES = nb04_manifest['model_safe_names']

print(f'X_train_smote : {X_train_smote.shape}')
print(f'X_train_pp    : {X_train_pp.shape}  (for CV)')
print(f'X_val_pp      : {X_val_pp.shape}')
print(f'Top 3 to tune : {TOP3}')
print()
print('NB04 BASELINES (val, threshold=0.50):')
for n in TOP3:
    r = nb04_val[n]
    print(f'  {n:<22}  F1={r["f1"]:.4f}  PR-AUC={r["pr_auc"]:.4f}')


In [ ]:
N_TRIALS    = 50   # ← increase to 100+ for production quality
CV_STRATEGY = StratifiedKFold(n_splits=cfg.CV_FOLDS, shuffle=True,
                               random_state=cfg.RANDOM_STATE)

neg_count         = int((y_train == 0).sum())
pos_count         = int((y_train == 1).sum())
SCALE_POS_W_TRAIN = neg_count / pos_count          # for CV objective

smote_neg          = int((y_train_smote == 0).sum())
smote_pos          = int((y_train_smote == 1).sum())
SCALE_POS_W_SMOTE  = smote_neg / smote_pos         # for SMOTE refit

print(f'N_TRIALS          : {N_TRIALS}')
print(f'scale_pos_weight  : {SCALE_POS_W_TRAIN:.3f} (CV)  /  {SCALE_POS_W_SMOTE:.3f} (refit)')


---
## 2. Optuna Hyperparameter Search

### Search Space Rationale

| Parameter | Range | Rationale |
|-----------|-------|-----------|
| `learning_rate` | log-uniform [0.01, 0.30] | Log scale balances fine and coarse regimes |
| `max_depth` | int [3, 9] | Shallow avoids overfit on ~40k samples; deep captures interactions |
| `n_estimators` | int [100, 400] | Diminishing returns beyond 300 |
| `subsample` / `colsample_bytree` | [0.60, 1.00] | Stochastic regularisation for boosting |
| `reg_alpha` / `reg_lambda` | L1 / L2 penalty | Sparsity on high-dimensional engineered features |
| LR `C` | log-uniform [1e-4, 100] | Covers strong regularisation to near-unregularised |


In [ ]:
def make_objective(model_name, X_tr, y_tr, cv_strategy, xgb_spw):
    """Returns f(trial) -> float for optuna.study.optimize()."""
    def objective(trial):
        if model_name == 'Logistic Regression':
            model = LogisticRegression(
                C      = trial.suggest_float('C', 1e-4, 100.0, log=True),
                solver = trial.suggest_categorical('solver', ['lbfgs', 'saga']),
                max_iter=1000, class_weight='balanced',
                random_state=cfg.RANDOM_STATE, n_jobs=1,
            )
        elif model_name == 'Random Forest':
            model = RandomForestClassifier(
                n_estimators     = trial.suggest_int('n_estimators', 100, 400, step=50),
                max_depth        = trial.suggest_int('max_depth', 4, 25),
                min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 25),
                max_features     = trial.suggest_categorical('max_features', ['sqrt', 'log2']),
                class_weight='balanced_subsample',
                random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS,
            )
        elif model_name == 'XGBoost':
            model = XGBClassifier(
                n_estimators     = trial.suggest_int('n_estimators', 100, 400, step=50),
                learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
                max_depth        = trial.suggest_int('max_depth', 3, 9),
                subsample        = trial.suggest_float('subsample', 0.60, 1.00),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.60, 1.00),
                min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
                gamma            = trial.suggest_float('gamma', 0.00, 0.50),
                reg_alpha        = trial.suggest_float('reg_alpha', 0.00, 2.00),
                reg_lambda       = trial.suggest_float('reg_lambda', 0.50, 3.00),
                scale_pos_weight=xgb_spw,
                tree_method='hist', verbosity=0,
                random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS,
            )
        elif model_name == 'LightGBM':
            model = LGBMClassifier(
                n_estimators     = trial.suggest_int('n_estimators', 100, 400, step=50),
                learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
                num_leaves       = trial.suggest_int('num_leaves', 20, 150),
                max_depth        = trial.suggest_int('max_depth', 3, 9),
                subsample        = trial.suggest_float('subsample', 0.60, 1.00),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.60, 1.00),
                min_child_samples= trial.suggest_int('min_child_samples', 10, 60),
                reg_alpha        = trial.suggest_float('reg_alpha', 0.00, 2.00),
                reg_lambda       = trial.suggest_float('reg_lambda', 0.50, 3.00),
                class_weight='balanced', verbose=-1,
                random_state=cfg.RANDOM_STATE, n_jobs=1,
            )
        else:
            raise ValueError(f'No search space for: {model_name}')

        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_strategy, scoring='f1',
                                 n_jobs=1, error_score=0.0)
        return float(scores.mean())
    return objective


def build_model_from_params(model_name, params, xgb_spw):
    """Reconstruct model with best Optuna params + production settings."""
    if model_name == 'Logistic Regression':
        return LogisticRegression(**params, max_iter=1000, class_weight='balanced',
                                  random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS)
    elif model_name == 'Random Forest':
        return RandomForestClassifier(**params, class_weight='balanced_subsample',
                                      random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS)
    elif model_name == 'XGBoost':
        return XGBClassifier(**params, scale_pos_weight=xgb_spw,
                             tree_method='hist', verbosity=0,
                             random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS)
    elif model_name == 'LightGBM':
        return LGBMClassifier(**params, class_weight='balanced', verbose=-1,
                              random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS)
    else:
        raise ValueError(f'Unknown model: {model_name}')


print('Objective factory and model builder defined.')


In [ ]:
# Wall-time estimate (50 trials, 5-fold CV, modern laptop):
#   Logistic Regression : ~1-2 min
#   Random Forest       : ~15-25 min
#   XGBoost             : ~8-15 min
#   LightGBM            : ~5-10 min

studies     = {}
best_params = {}
study_times = {}

for name in TOP3:
    print(f'\n{"="*60}')
    print(f'  Tuning: {name}  ({N_TRIALS} trials x {cfg.CV_FOLDS} folds)')
    print(f'{"="*60}')
    t0      = time.time()
    sampler = TPESampler(seed=cfg.RANDOM_STATE)
    study   = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(
        make_objective(name, X_train_pp, y_train, CV_STRATEGY, SCALE_POS_W_TRAIN),
        n_trials=N_TRIALS,
        show_progress_bar=True,
    )
    elapsed = time.time() - t0
    studies[name]     = study
    best_params[name] = study.best_params
    study_times[name] = round(elapsed, 1)

    delta = study.best_value - nb04_val[name]['f1']
    print(f'  Best CV F1 (tuned)  : {study.best_value:.4f}')
    print(f'  NB04 baseline F1    : {nb04_val[name]["f1"]:.4f}')
    print(f'  Delta               : {delta:+.4f}')
    print(f'  Time                : {elapsed:.0f}s')
    print(f'  Best params         : {study.best_params}')

print('\nAll studies complete.')


In [ ]:
print('BEST HYPERPARAMETERS FOUND BY OPTUNA')
print('=' * 60)
for name in TOP3:
    study = studies[name]
    print(f'\n  {name}')
    print(f'  Best CV F1 : {study.best_value:.4f}  (baseline: {nb04_val[name]["f1"]:.4f})')
    for k, v in sorted(study.best_params.items()):
        fmt = f'{v:.5f}' if isinstance(v, float) else str(v)
        print(f'    {k:<25}: {fmt}')


In [ ]:
n_m = len(TOP3)
fig, axes = plt.subplots(1, n_m, figsize=(7 * n_m, 5))
if n_m == 1:
    axes = [axes]

for ax, name in zip(axes, TOP3):
    tdf = studies[name].trials_dataframe()
    tdf = tdf[tdf['state'] == 'COMPLETE'].sort_values('number')
    ax.scatter(tdf['number'], tdf['value'], alpha=0.35, s=18, color='#90CAF9', zorder=2)
    ax.plot(tdf['number'], tdf['value'].cummax(),
            color='#E53935', linewidth=2.0, label='Best-so-far', zorder=3)
    ax.axhline(nb04_val[name]['f1'], color='#9E9E9E', linestyle='--', linewidth=1.5,
               label=f'NB04 baseline ({nb04_val[name]["f1"]:.3f})')
    ax.axhline(0.70, color='#FF9800', linestyle=':', linewidth=1.5, label='Target F1=0.70')
    ax.set_title(f'{name}\nBest: {studies[name].best_value:.4f}', fontweight='bold')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('5-Fold CV F1')
    ax.legend(fontsize=8)

plt.suptitle('Optuna Optimisation History — CV F1 per Trial',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb05_optuna_history.png',
            bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, n_m, figsize=(7 * n_m, 5))
if n_m == 1:
    axes = [axes]

for ax, name in zip(axes, TOP3):
    study = studies[name]
    try:
        imp = optuna.importance.get_param_importances(
            study,
            evaluator=optuna.importance.FanovaImportanceEvaluator(seed=cfg.RANDOM_STATE),
        )
    except Exception:
        # Fallback: Pearson correlation between param values and objective
        df = study.trials_dataframe()
        df = df[df['state'] == 'COMPLETE']
        pcols = [c for c in df.columns if c.startswith('params_')]
        imp = {
            c.replace('params_', ''): abs(df[c].corr(df['value']))
            for c in pcols if c in df.columns
        }
        imp = dict(sorted(imp.items(), key=lambda x: -x[1]))

    keys = list(imp.keys())
    vals = [imp[k] for k in keys]
    ax.barh(keys, vals, color='#42A5F5', edgecolor='white')
    for i, (k, v) in enumerate(zip(keys, vals)):
        ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=8)
    ax.set_title(f'{name}\nParameter Importance', fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.suptitle('Hyperparameter Importance — Which Parameters Matter Most?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb05_param_importance.png',
            bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()


---
## 3. Refit Tuned Models on SMOTE Training Data

Best params found by Optuna (CV on raw training) are used to build final models.
These are then fitted on the **full SMOTE-balanced training set** for maximum data utilisation.


In [ ]:
tuned_models = {}
print(f'Refitting {len(TOP3)} models on X_train_smote {X_train_smote.shape}...')
print()
for name in TOP3:
    t0    = time.time()
    model = build_model_from_params(name, best_params[name], SCALE_POS_W_SMOTE)
    model.fit(X_train_smote, y_train_smote)
    elapsed = time.time() - t0
    tuned_models[name] = model
    y_prob   = model.predict_proba(X_val_pp)[:, 1]
    quick_f1 = f1_score(y_val, (y_prob >= 0.5).astype(int), pos_label=1, zero_division=0)
    quick_ap = average_precision_score(y_val, y_prob)
    print(f'  {name:<22}  {elapsed:.1f}s   val_F1@0.5={quick_f1:.4f}   PR-AUC={quick_ap:.4f}')
print('\nAll tuned models fitted.')


---
## 4. Validation Comparison: Baseline vs Tuned

In [ ]:
tuned_val = {}
tuned_thr = {}

for name, model in tuned_models.items():
    safe  = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    print(f'\nEvaluating: {name} (Tuned)')
    result = evaluate_model(
        model, X_val_pp, y_val,
        model_name=f'{name} (Tuned)',
        threshold=0.50,
        save_path=cfg.FIGURES_DIR / f'nb05_eval_{safe}_tuned.png',
    )
    tuned_val[name] = result
    opt_thr, opt_f1 = find_optimal_threshold(model, X_val_pp, y_val, metric='f1')
    tuned_thr[name] = {'optimal_thr': opt_thr, 'optimal_f1': opt_f1}
    result.update({'optimal_threshold': opt_thr, 'optimal_f1': opt_f1})

print('\nEvaluation complete.')


In [ ]:
rows = []
for name in TOP3:
    b, t   = nb04_val[name], tuned_val[name]
    bt, tt = nb04_thr[name], tuned_thr[name]
    rows.append({
        'Model':           name,
        'F1@0.5 base':     b['f1'],
        'F1@0.5 tuned':    t['f1'],
        'dF1@0.5':         round(t['f1']  - b['f1'], 4),
        'F1@opt base':     bt['optimal_f1'],
        'F1@opt tuned':    tt['optimal_f1'],
        'dF1@opt':         round(tt['optimal_f1'] - bt['optimal_f1'], 4),
        'PR-AUC base':     b['pr_auc'],
        'PR-AUC tuned':    t['pr_auc'],
        'dPR-AUC':         round(t['pr_auc'] - b['pr_auc'], 4),
        'Opt.Thr base':    bt['optimal_thr'],
        'Opt.Thr tuned':   tt['optimal_thr'],
    })
compare_df = pd.DataFrame(rows)
print('BASELINE (NB04) vs TUNED (NB05) — VALIDATION SET')
print('=' * 80)
print(compare_df.T.to_string())
print()
print('Positive delta = tuning improved the model.')


In [ ]:
metrics   = ['F1@0.5',      'F1@opt',           'PR-AUC']
base_cols = ['F1@0.5 base', 'F1@opt base',       'PR-AUC base']
tune_cols = ['F1@0.5 tuned','F1@opt tuned',      'PR-AUC tuned']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(TOP3))
w = 0.35
short = [n.replace('Random Forest', 'RF').replace('Logistic Regression', 'LR')
          .replace('LightGBM', 'LGBM').replace('XGBoost', 'XGB') for n in TOP3]

for ax, metric, bc, tc in zip(axes, metrics, base_cols, tune_cols):
    bv = compare_df[bc].tolist()
    tv = compare_df[tc].tolist()
    b1 = ax.bar(x - w/2, bv, w, label='NB04 Baseline', color='#90CAF9', edgecolor='white')
    b2 = ax.bar(x + w/2, tv, w, label='NB05 Tuned',    color='#E53935', edgecolor='white')
    ax.axhline(0.70, color='#FF9800', linestyle='--', linewidth=1.5, label='Target 0.70')
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.003,
                    f'{h:.3f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(short)
    ax.set_ylim(0, 1.0)
    ax.set_title(metric, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('NB04 Baseline vs NB05 Tuned — Validation Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb05_tuning_improvement.png',
            bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()


---
## 5. Save Tuned Models & Results

In [ ]:
# Models
for name, model in tuned_models.items():
    safe = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    out  = cfg.MODELS_DIR / f'{safe}_tuned.joblib'
    joblib.dump(model, out)
    print(f'  Saved: {out.name}')

# JSON artefacts
with open(PROCESSED / 'best_params_nb05.json', 'w') as f:
    json.dump(best_params, f, indent=2)
with open(PROCESSED / 'val_results_nb05.json', 'w') as f:
    json.dump(tuned_val, f, indent=2)
with open(PROCESSED / 'threshold_results_nb05.json', 'w') as f:
    json.dump(tuned_thr, f, indent=2)

compare_df.to_csv(PROCESSED / 'tuning_comparison_nb05.csv', index=False)

# NB06 manifest: best model by validation PR-AUC
best_name = max(TOP3, key=lambda n: tuned_val[n]['pr_auc'])
best_safe = SAFE_NAMES.get(best_name, best_name.lower().replace(' ', '_'))

nb05_manifest = {
    'tuned_models':        TOP3,
    'best_model_name':     best_name,
    'best_model_file':     f'{best_safe}_tuned.joblib',
    'best_val_pr_auc':     tuned_val[best_name]['pr_auc'],
    'best_val_f1_opt':     tuned_thr[best_name]['optimal_f1'],
    'best_opt_threshold':  tuned_thr[best_name]['optimal_thr'],
    'model_safe_names':    SAFE_NAMES,
    'n_trials':            N_TRIALS,
}
with open(PROCESSED / 'nb05_manifest.json', 'w') as f:
    json.dump(nb05_manifest, f, indent=2)

print()
print(f'Best model for NB06 : {best_name}')
print(f'  PR-AUC (val)      : {tuned_val[best_name]["pr_auc"]:.4f}')
print(f'  F1 @ opt thr      : {tuned_thr[best_name]["optimal_f1"]:.4f}')
print(f'  Optimal threshold : {tuned_thr[best_name]["optimal_thr"]:.3f}')


In [ ]:
print('NB05 ARTIFACT INVENTORY')
print('Models (models/):')
for name in tuned_models:
    safe = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    p    = cfg.MODELS_DIR / f'{safe}_tuned.joblib'
    if p.exists():
        print(f'  {p.name}: {p.stat().st_size / 1024:.0f} KB')

print('\nResults (data/processed/):')
for fname in ['best_params_nb05.json', 'val_results_nb05.json',
              'threshold_results_nb05.json', 'nb05_manifest.json',
              'tuning_comparison_nb05.csv']:
    p = PROCESSED / fname
    if p.exists():
        print(f'  {fname}: {p.stat().st_size / 1024:.1f} KB')

print('\nFigures (reports/figures/):')
for f in sorted(cfg.FIGURES_DIR.glob('nb05_*.png')):
    print(f'  {f.name}')


---
## 6. Summary

| Step | Method | Key Point |
|------|--------|-----------|
| Sampler | TPE | Bayesian search; improves with each trial |
| Objective | 5-fold CV F1, raw training | No SMOTE leakage across CV folds |
| Budget | 50 trials / model | Reaches ~90% of optimum |
| Final refit | SMOTE-balanced training | Full data utilisation |
| Best model | Highest validation PR-AUC | Passed to NB06 for test evaluation |

### Next — NB06: Final Evaluation + SHAP Explainability
- **First and only use of the test set**
- SHAP global and local explanations
- New Brunswick subgroup analysis
- Power BI CSV export for dashboard creation
